In [3]:
# Import

import pandas as pd
import numpy as np
from scipy.optimize import minimize
from scipy.special import comb

# Konstansok
TOTAL_NUMBERS = 90
NUMBERS_DRAWN = 5

# Kombinációs valószínűségek
TOTAL_COMBINATIONS = comb(90, 5, exact=True)
P_5 = 1 / TOTAL_COMBINATIONS
P_4 = comb(5, 4) * comb(85, 1) / TOTAL_COMBINATIONS
P_3 = comb(5, 3) * comb(85, 2) / TOTAL_COMBINATIONS
P_2 = comb(5, 2) * comb(85, 3) / TOTAL_COMBINATIONS

print(f"Valószínűségek:")
print(f"5 találat: 1/{TOTAL_COMBINATIONS:,} = {P_5:.10f}")
print(f"4 találat: {P_4:.6f}")
print(f"3 találat: {P_3:.6f}")
print(f"2 találat: {P_2:.6f}")

Valószínűségek:
5 találat: 1/43,949,268 = 0.0000000228
4 találat: 0.000010
3 találat: 0.000812
2 találat: 0.022474


In [4]:
# ADATOK BETÖLTÉSE ÉS TISZTÍTÁSA

huzasok = pd.read_csv("otos.csv", sep=";", header=None, 
                      names=["year", "week", "date", "fives", "fives_prize", 
                             "fours", "fours_prize", "threes", "threes_prize", 
                             "twos", "twos_prize", "n1", "n2", "n3", "n4", "n5"])

# Szelvényárak
szelo_arak = {
    '1957.01.': 130,
    '2003.04.': 150,
    '2005.03.': 175,
    '2007.11.': 200,
    '2010.02.': 225,
    '2016.11.': 250,
    '2020.01.': 300,
    '2022.10.': 350,
    '2023.09.': 400
}

def get_ticket_price(date_str):
    """Meghatározza a szelvény árát az adott dátumra"""
    if pd.isna(date_str):
        return 130
    
    for date_threshold, price in sorted(szelo_arak.items(), reverse=True):
        year = int(date_threshold.split('.')[0])
        month = int(date_threshold.split('.')[1])
        
        try:
            draw_year = int(date_str.split('.')[0])
            draw_month = int(date_str.split('.')[1])
            
            if draw_year > year or (draw_year == year and draw_month >= month):
                return price
        except:
            return 130
    
    return 130

# Szelvényár hozzáadása
huzasok['ticket_price'] = huzasok['date'].apply(get_ticket_price)

# Nyeremények tisztítása (eltávolítjuk az "Ft" és szóközöket, átalakítjuk számmá)
def clean_prize(prize_str):
    if pd.isna(prize_str) or prize_str == '0 Ft' or prize_str == 0:
        return 0
    try:
        return int(str(prize_str).replace('Ft', '').replace(' ', '').replace(',', ''))
    except:
        return 0

for col in ['fives_prize', 'fours_prize', 'threes_prize', 'twos_prize']:
    huzasok[col] = huzasok[col].apply(clean_prize)

# Nyeremények relativizálása (hányszorosa a szelvényárnak)
huzasok['rel_fives_prize'] = huzasok['fives_prize'] / huzasok['ticket_price']
huzasok['rel_fours_prize'] = huzasok['fours_prize'] / huzasok['ticket_price']
huzasok['rel_threes_prize'] = huzasok['threes_prize'] / huzasok['ticket_price']
huzasok['rel_twos_prize'] = huzasok['twos_prize'] / huzasok['ticket_price']

# Csak azokat hagyjuk, ahol van adat
huzasok_clean = huzasok[
    (huzasok['twos'] > 0) & 
    (huzasok['threes'] > 0) & 
    (huzasok['fours'] > 0)
].copy()

print(f"\n\nÖsszesen {len(huzasok)} sorsolás, ebből felhasználható: {len(huzasok_clean)}")



Összesen 3586 sorsolás, ebből felhasználható: 1456


In [5]:
# JÁTÉKOSSZÁM BECSLÉSE

print("\n" + "="*80)
print("JÁTÉKOSSZÁM BECSLÉSE")
print("="*80)

def estimate_players_from_category(winners, probability):
    """Játékosszám becslése egy kategóriából"""
    if winners == 0:
        return np.nan
    return winners / probability

# Becslés minden kategóriából
huzasok_clean['players_from_2'] = huzasok_clean['twos'].apply(
    lambda x: estimate_players_from_category(x, P_2))
huzasok_clean['players_from_3'] = huzasok_clean['threes'].apply(
    lambda x: estimate_players_from_category(x, P_3))
huzasok_clean['players_from_4'] = huzasok_clean['fours'].apply(
    lambda x: estimate_players_from_category(x, P_4))

# Súlyozott átlag (2-találatost preferáljuk, mert nagy mintaszám és stabil)
huzasok_clean['estimated_players'] = (
    huzasok_clean['players_from_2'] * 0.7 +
    huzasok_clean['players_from_3'] * 0.2 +
    huzasok_clean['players_from_4'] * 0.1
)

# Outlierek szűrése (pl. IQR módszer)
Q1 = huzasok_clean['estimated_players'].quantile(0.25)
Q3 = huzasok_clean['estimated_players'].quantile(0.75)
IQR = Q3 - Q1
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

huzasok_clean['estimated_players'] = huzasok_clean['estimated_players'].clip(
    lower=lower_bound, upper=upper_bound)

print(f"\nBecsült játékosszám statisztikák:")
print(huzasok_clean['estimated_players'].describe())

# Időbeli trend
recent_data = huzasok_clean[huzasok_clean['year'] >= 2020]
print(f"\n2020 óta átlagos játékosszám: {recent_data['estimated_players'].mean():,.0f}")


JÁTÉKOSSZÁM BECSLÉSE

Becsült játékosszám statisztikák:
count    1.456000e+03
mean     4.249566e+06
std      1.498286e+06
min      1.767205e+06
25%      3.123558e+06
50%      3.929722e+06
75%      5.044678e+06
max      7.926357e+06
Name: estimated_players, dtype: float64

2020 óta átlagos játékosszám: 3,295,396


In [7]:
# SZÁMNÉPSZERŰSÉG BECSLÉSE

# ============================================================================
# SZÁMNÉPSZERŰSÉG BECSLÉSE - JAVÍTOTT VERZIÓ
# ============================================================================

print("\n" + "="*80)
print("SZÁMNÉPSZERŰSÉG BECSLÉSE (2, 3, 4 TALÁLATOSOK ALAPJÁN)")
print("="*80)

def calculate_number_popularity_from_category(huzasok_df, category='twos'):
    """
    Számnépszerűség becslése egy találati kategória alapján
    
    category: 'twos', 'threes', vagy 'fours'
    """
    
    # Kategória adatai
    if category == 'twos':
        winners_col = 'twos'
        probability = P_2
        match_count = 2
    elif category == 'threes':
        winners_col = 'threes'
        probability = P_3
        match_count = 3
    elif category == 'fours':
        winners_col = 'fours'
        probability = P_4
        match_count = 4
    else:
        raise ValueError("Invalid category")
    
    # Csak azokat a sorsolásokat, ahol van nyertes
    huzasok_cat = huzasok_df[huzasok_df[winners_col] > 0].copy()
    
    print(f"\n{category.upper()}: {len(huzasok_cat)} sorsolás elemezve")
    
    # Várható nyertesek száma, ha minden kombináció egyformán népszerű lenne
    huzasok_cat['expected_winners'] = (
        huzasok_cat['estimated_players'] * probability
    )
    
    # Megfigyelt / várható arány
    huzasok_cat['popularity_factor'] = (
        huzasok_cat[winners_col] / huzasok_cat['expected_winners']
    )
    
    # Outlier szűrés (túl extrém értékek)
    Q1 = huzasok_cat['popularity_factor'].quantile(0.1)
    Q3 = huzasok_cat['popularity_factor'].quantile(0.9)
    huzasok_cat = huzasok_cat[
        (huzasok_cat['popularity_factor'] >= Q1) & 
        (huzasok_cat['popularity_factor'] <= Q3)
    ]
    
    print(f"  Outlier szűrés után: {len(huzasok_cat)} sorsolás")
    print(f"  Átlagos popularity_factor: {huzasok_cat['popularity_factor'].mean():.3f}")
    
    # Számonkénti elemzés
    number_scores = {}
    
    for num in range(1, 91):
        # Sorsolások, ahol ez a szám benne volt a kihúzott 5-ben
        with_num = huzasok_cat[
            (huzasok_cat['n1'] == num) |
            (huzasok_cat['n2'] == num) |
            (huzasok_cat['n3'] == num) |
            (huzasok_cat['n4'] == num) |
            (huzasok_cat['n5'] == num)
        ]
        
        if len(with_num) >= 10:  # Minimum 10 előfordulás kell
            # Átlagos popularity factor, amikor ez a szám benne van
            avg_with = with_num['popularity_factor'].mean()
            # Átlagos popularity factor általában
            avg_overall = huzasok_cat['popularity_factor'].mean()
            
            # Relatív népszerűség
            number_scores[num] = avg_with / avg_overall
        else:
            number_scores[num] = None
    
    return number_scores, len(huzasok_cat)

# ============================================================================
# NÉPSZERŰSÉG BECSLÉSE MINDEN KATEGÓRIÁBÓL
# ============================================================================

popularity_twos, count_twos = calculate_number_popularity_from_category(
    huzasok_clean, 'twos')
popularity_threes, count_threes = calculate_number_popularity_from_category(
    huzasok_clean, 'threes')
popularity_fours, count_fours = calculate_number_popularity_from_category(
    huzasok_clean, 'fours')

# ============================================================================
# KOMBINÁLT NÉPSZERŰSÉG (SÚLYOZOTT ÁTLAG)
# ============================================================================

print("\n" + "="*80)
print("KOMBINÁLT NÉPSZERŰSÉG SZÁMÍTÁSA")
print("="*80)

# Súlyok: 2-találat a legtöbb adat, 4-találat a legkevesebb de legpontosabb
weights = {
    'twos': 0.5,      # Legnagyobb mintaszám
    'threes': 0.3,    # Közepes mintaszám
    'fours': 0.2      # Kis mintaszám, de direkt kapcsolat
}

number_popularity = {}

for num in range(1, 91):
    scores = []
    score_weights = []
    
    if popularity_twos[num] is not None:
        scores.append(popularity_twos[num])
        score_weights.append(weights['twos'])
    
    if popularity_threes[num] is not None:
        scores.append(popularity_threes[num])
        score_weights.append(weights['threes'])
    
    if popularity_fours[num] is not None:
        scores.append(popularity_fours[num])
        score_weights.append(weights['fours'])
    
    if scores:
        # Súlyozott átlag
        number_popularity[num] = np.average(scores, weights=score_weights)
    else:
        # Ha nincs elég adat, semleges 1.0
        number_popularity[num] = 1.0

# Normalizálás (átlag pontosan 1.0 legyen)
mean_pop = np.mean(list(number_popularity.values()))
number_popularity = {k: v/mean_pop for k, v in number_popularity.items()}

print(f"\nSzámonkénti népszerűség kiszámítva {len(number_popularity)} számra")
print(f"Átlag népszerűség (normalizálás után): {np.mean(list(number_popularity.values())):.6f}")
print(f"Szórás: {np.std(list(number_popularity.values())):.6f}")

# ============================================================================
# EREDMÉNYEK MEGJELENÍTÉSE
# ============================================================================

sorted_by_pop = sorted(number_popularity.items(), key=lambda x: x[1], reverse=True)

print("\n" + "="*80)
print("Top 15 LEGnépszerűbb szám:")
print("="*80)
for i, (num, pop) in enumerate(sorted_by_pop[:15], 1):
    # Részletes info
    info_parts = []
    if popularity_twos[num]:
        info_parts.append(f"2:{popularity_twos[num]:.3f}")
    if popularity_threes[num]:
        info_parts.append(f"3:{popularity_threes[num]:.3f}")
    if popularity_fours[num]:
        info_parts.append(f"4:{popularity_fours[num]:.3f}")
    
    info_str = ", ".join(info_parts)
    print(f"{i:2d}. Szám {num:2d}: {pop:.4f}× ({info_str})")

print("\n" + "="*80)
print("Top 15 LEGkevésbé népszerű szám:")
print("="*80)
for i, (num, pop) in enumerate(sorted_by_pop[-15:], 1):
    # Részletes info
    info_parts = []
    if popularity_twos[num]:
        info_parts.append(f"2:{popularity_twos[num]:.3f}")
    if popularity_threes[num]:
        info_parts.append(f"3:{popularity_threes[num]:.3f}")
    if popularity_fours[num]:
        info_parts.append(f"4:{popularity_fours[num]:.3f}")
    
    info_str = ", ".join(info_parts)
    print(f"{i:2d}. Szám {num:2d}: {pop:.4f}× ({info_str})")

# ============================================================================
# MINTÁZATOK ELEMZÉSE
# ============================================================================

print("\n" + "="*80)
print("MINTÁZATOK ELEMZÉSE")
print("="*80)

# Különböző tartományok
ranges = {
    '1-10': range(1, 11),
    '11-20': range(11, 21),
    '21-31 (születésnap vége)': range(21, 32),
    '32-40': range(32, 41),
    '41-50': range(41, 51),
    '51-60': range(51, 61),
    '61-70': range(61, 71),
    '71-80': range(71, 81),
    '81-90': range(81, 91),
}

print("\nNépszerűség tartományonként:")
for range_name, num_range in ranges.items():
    avg_pop = np.mean([number_popularity[n] for n in num_range])
    print(f"  {range_name:25s}: {avg_pop:.4f}×")

# Speciális csoportok
special_groups = {
    'Születésnapok (1-31)': range(1, 32),
    'Magas számok (60-90)': range(60, 91),
    'Kerek számok (×10)': [10, 20, 30, 40, 50, 60, 70, 80, 90],
    'Prímek (első 15)': [2, 3, 5, 7, 11, 13, 17, 19, 23, 29, 31, 37, 41, 43, 47],
    'Páros számok': range(2, 91, 2),
    'Páratlan számok': range(1, 91, 2),
}

print("\nSpeciális csoportok:")
for group_name, numbers in special_groups.items():
    avg_pop = np.mean([number_popularity[n] for n in numbers])
    print(f"  {group_name:25s}: {avg_pop:.4f}×")

# Vizuális megjelenítés
print("\n" + "="*80)
print("HŐTÉRKÉP (számonkénti népszerűség)")
print("="*80)
print("\n1-45:")
for i in range(0, 45, 9):
    row = []
    for num in range(i+1, min(i+10, 46)):
        pop = number_popularity[num]
        if pop > 1.05:
            symbol = "██"  # Nagyon népszerű
        elif pop > 1.02:
            symbol = "▓▓"  # Népszerű
        elif pop > 0.98:
            symbol = "▒▒"  # Átlagos
        elif pop > 0.95:
            symbol = "░░"  # Kevésbé népszerű
        else:
            symbol = "  "  # Ritka
        row.append(f"{num:2d}{symbol}")
    print("  ".join(row))

print("\n46-90:")
for i in range(45, 90, 9):
    row = []
    for num in range(i+1, min(i+10, 91)):
        pop = number_popularity[num]
        if pop > 1.05:
            symbol = "██"
        elif pop > 1.02:
            symbol = "▓▓"
        elif pop > 0.98:
            symbol = "▒▒"
        elif pop > 0.95:
            symbol = "░░"
        else:
            symbol = "  "
        row.append(f"{num:2d}{symbol}")
    print("  ".join(row))

print("\nJelmagyarázat: ██ = nagyon népszerű, ▓▓ = népszerű, ▒▒ = átlagos, ░░ = kevésbé népszerű,    = ritka")

# ============================================================================
# KONZISZTENCIA ELLENŐRZÉS
# ============================================================================

print("\n" + "="*80)
print("KONZISZTENCIA ELLENŐRZÉS")
print("="*80)

# Mennyire korrelálnak a különböző kategóriák eredményei?
from scipy.stats import spearmanr

# Csak azok a számok, ahol mindhárom kategóriában van adat
common_numbers = [
    num for num in range(1, 91) 
    if all([
        popularity_twos[num] is not None,
        popularity_threes[num] is not None,
        popularity_fours[num] is not None
    ])
]

if len(common_numbers) > 20:
    scores_2 = [popularity_twos[n] for n in common_numbers]
    scores_3 = [popularity_threes[n] for n in common_numbers]
    scores_4 = [popularity_fours[n] for n in common_numbers]
    
    corr_23, p_23 = spearmanr(scores_2, scores_3)
    corr_24, p_24 = spearmanr(scores_2, scores_4)
    corr_34, p_34 = spearmanr(scores_3, scores_4)
    
    print(f"\nSpearman korreláció ({len(common_numbers)} közös szám):")
    print(f"  2-találat vs 3-találat: {corr_23:.3f} (p={p_23:.4f})")
    print(f"  2-találat vs 4-találat: {corr_24:.3f} (p={p_24:.4f})")
    print(f"  3-találat vs 4-találat: {corr_34:.3f} (p={p_34:.4f})")
    
    if all([corr_23 > 0.3, corr_24 > 0.3, corr_34 > 0.3]):
        print("  ✓ Konzisztens becslés - a különböző kategóriák hasonló mintázatot mutatnak")
    else:
        print("  ⚠ Alacsony korreláció - a becslés bizonytalanabb")

print("\n" + "="*80)
print("NÉPSZERŰSÉG BECSLÉS KÉSZ")
print("="*80)


SZÁMNÉPSZERŰSÉG BECSLÉSE (2, 3, 4 TALÁLATOSOK ALAPJÁN)

TWOS: 1456 sorsolás elemezve
  Outlier szűrés után: 1164 sorsolás
  Átlagos popularity_factor: 1.021

THREES: 1456 sorsolás elemezve
  Outlier szűrés után: 1164 sorsolás
  Átlagos popularity_factor: 0.985

FOURS: 1456 sorsolás elemezve
  Outlier szűrés után: 1164 sorsolás
  Átlagos popularity_factor: 0.923

KOMBINÁLT NÉPSZERŰSÉG SZÁMÍTÁSA

Számonkénti népszerűség kiszámítva 90 számra
Átlag népszerűség (normalizálás után): 1.000000
Szórás: 0.008911

Top 15 LEGnépszerűbb szám:
 1. Szám  5: 1.0225× (2:0.985, 3:1.026, 4:1.111)
 2. Szám 19: 1.0210× (2:0.985, 3:1.028, 4:1.100)
 3. Szám 73: 1.0203× (2:0.993, 3:1.008, 4:1.106)
 4. Szám  4: 1.0164× (2:0.985, 3:1.024, 4:1.081)
 5. Szám  9: 1.0152× (2:0.988, 3:1.020, 4:1.076)
 6. Szám 33: 1.0125× (2:0.987, 3:1.021, 4:1.062)
 7. Szám 49: 1.0123× (2:0.992, 3:1.013, 4:1.061)
 8. Szám 68: 1.0104× (2:0.998, 3:1.017, 4:1.030)
 9. Szám 21: 1.0104× (2:0.993, 3:1.025, 4:1.032)
10. Szám 13: 1.0089× (

In [9]:
# VÁRHATÓ ÉRTÉK KALKULÁCIÓ

print("\n" + "="*80)
print("VÁRHATÓ ÉRTÉK KALKULÁCIÓ")
print("="*80)

# Átlagos nyeremények (relativizált) - csak pozitív értékek
avg_prize_4 = huzasok_clean[huzasok_clean['rel_fours_prize'] > 0]['rel_fours_prize'].mean()
avg_prize_3 = huzasok_clean[huzasok_clean['rel_threes_prize'] > 0]['rel_threes_prize'].mean()
avg_prize_2 = huzasok_clean[huzasok_clean['rel_twos_prize'] > 0]['rel_twos_prize'].mean()

print(f"\nÁtlagos nyeremények (szelvényár többszöröse):")
print(f"  4 találat: {avg_prize_4:.1f}× (kb {avg_prize_4 * 400:,.0f} Ft mai áron)")
print(f"  3 találat: {avg_prize_3:.1f}× (kb {avg_prize_3 * 400:,.0f} Ft mai áron)")
print(f"  2 találat: {avg_prize_2:.1f}× (kb {avg_prize_2 * 400:,.0f} Ft mai áron)")

# 5-találat becslése
# Problémák a huzasok_with_5 használatával:
# 1. Csak azokat látjuk, ahol VOLT nyertes (bias)
# 2. Nem látjuk a felhalmozódott jackpotokat
# 3. Kis mintaszám

# JOBB MEGKÖZELÍTÉS: Becsüljük a teljes nyereményalapot
# A Szerencsejáték Zrt. kb. 50%-ot fizet vissza nyereményekben

print("\n" + "-"*80)
print("5-TALÁLAT JACKPOT BECSLÉSE")
print("-"*80)

# Számítsuk ki a teljes befizetést és nyereménykiadást
huzasok_recent = huzasok_clean[huzasok_clean['year'] >= 2020].copy()
print(f"\nElemzés 2020 óta: {len(huzasok_recent)} sorsolás")

# Átlagos játékosszám
avg_players_recent = huzasok_recent['estimated_players'].mean()
print(f"Átlagos játékosszám: {avg_players_recent:,.0f}")

# Teljes befizetés egy sorsoláson (relativizált, szelvényár többszöröse)
total_revenue_per_draw = avg_players_recent  # minden szelvény = 1× a szelvényár

# Kifizetési arány becslése a megfigyelt nyereményekből
# Számítsuk ki, mennyi ment 2, 3, 4 találatosokra
def calculate_payout_for_category(df, winners_col, prize_col, probability):
    """Átlagos kifizetés egy kategóriára (szelvényár többszöröse)"""
    total_payout = (df[winners_col] * df[prize_col]).sum()
    total_tickets = df['estimated_players'].sum()
    payout_per_ticket = total_payout / total_tickets
    return payout_per_ticket

payout_2 = calculate_payout_for_category(huzasok_recent, 'twos', 'rel_twos_prize', P_2)
payout_3 = calculate_payout_for_category(huzasok_recent, 'threes', 'rel_threes_prize', P_3)
payout_4 = calculate_payout_for_category(huzasok_recent, 'fours', 'rel_fours_prize', P_4)

print(f"\nÁtlagos kifizetés szelvényenként:")
print(f"  2 találat: {payout_2:.6f}× ({payout_2*100:.4f}%)")
print(f"  3 találat: {payout_3:.6f}× ({payout_3*100:.4f}%)")
print(f"  4 találat: {payout_4:.6f}× ({payout_4*100:.4f}%)")
print(f"  2+3+4 összesen: {(payout_2+payout_3+payout_4):.6f}× ({(payout_2+payout_3+payout_4)*100:.2f}%)")

# Ha feltételezzük, hogy ~50% a kifizetési arány, akkor a maradék 5-találatra megy
assumed_total_payout_ratio = 0.50
remaining_for_jackpot = assumed_total_payout_ratio - (payout_2 + payout_3 + payout_4)

print(f"\nHa a teljes kifizetési arány {assumed_total_payout_ratio*100:.0f}%:")
print(f"  5-találatra marad: {remaining_for_jackpot:.6f}× ({remaining_for_jackpot*100:.2f}%)")

# Ez az "átlagos" jackpot egy sorsoláson
estimated_avg_jackpot_per_draw = remaining_for_jackpot * avg_players_recent

print(f"\n→ Becsült átlagos teljes jackpot/sorsolás: {estimated_avg_jackpot_per_draw:.1f}×")
print(f"  (kb {estimated_avg_jackpot_per_draw * 400:,.0f} Ft mai áron)")

# Validálás: nézzük meg azokat a sorsolásokat, ahol volt 5-találatos
huzasok_with_5_recent = huzasok_recent[huzasok_recent['fives'] > 0].copy()
if len(huzasok_with_5_recent) > 0:
    # Teljes jackpot = nyertesek × nyeremény/fő
    huzasok_with_5_recent['total_jackpot'] = (
        huzasok_with_5_recent['fives'] * huzasok_with_5_recent['rel_fives_prize']
    )
    observed_avg_jackpot = huzasok_with_5_recent['total_jackpot'].mean()
    avg_winners = huzasok_with_5_recent['fives'].mean()
    
    print(f"\nValidálás ({len(huzasok_with_5_recent)} sorsolás volt 5-találatossal):")
    print(f"  Megfigyelt átlag jackpot: {observed_avg_jackpot:.1f}×")
    print(f"  Átlagos nyertesszám: {avg_winners:.2f}")
    print(f"  Különbség: {abs(observed_avg_jackpot - estimated_avg_jackpot_per_draw):.1f}× "
          f"({abs(observed_avg_jackpot - estimated_avg_jackpot_per_draw)/estimated_avg_jackpot_per_draw*100:.1f}%)")
    
    # Használjuk a megfigyelt értéket, ha van elég adat
    if len(huzasok_with_5_recent) > 20:
        print(f"  → Megfigyelt érték használata (elég nagy minta)")
        estimated_total_jackpot = observed_avg_jackpot
    else:
        print(f"  → Becsült érték használata (kis minta: {len(huzasok_with_5_recent)})")
        estimated_total_jackpot = estimated_avg_jackpot_per_draw
else:
    print(f"\nNincs 5-találatos a legutóbbi adatokban, becslést használjuk")
    estimated_total_jackpot = estimated_avg_jackpot_per_draw

print(f"\n{'='*80}")
print(f"VÉGSŐ BECSLÉS - Átlagos teljes jackpot: {estimated_total_jackpot:.1f}×")
print(f"                (kb {estimated_total_jackpot * 400:,.0f} Ft mai áron)")
print(f"{'='*80}")

# ============================================================================
# VÁRHATÓ ÉRTÉK FÜGGVÉNY
# ============================================================================

def calculate_ev(numbers, number_popularity, avg_players, 
                 jackpot_multiplier=1.0, verbose=False):
    """
    Várható érték számítása egy adott számkombinációra
    
    Parameters:
    -----------
    numbers : list
        5 szám listája (1-90)
    number_popularity : dict
        Számonkénti népszerűség (1.0 = átlag)
    avg_players : float
        Átlagos játékosszám egy sorsoláson
    jackpot_multiplier : float
        Jackpot szorzó (ha felhalmozódott)
    verbose : bool
        Részletes kiírás
    
    Returns:
    --------
    dict: Várható érték komponensek
    """
    
    # Népszerűségi faktor ennek a kombinációnak
    # Feltétel: a számok népszerűsége független (szorzat modell)
    combo_popularity = np.prod([number_popularity[n] for n in numbers])
    
    # Becsült játékosszám ezzel a PONTOS kombinációval
    # P(valaki ezt a kombinációt játssza) = 1/C(90,5) × népszerűség
    base_rate = avg_players / TOTAL_COMBINATIONS
    expected_players_with_combo = base_rate * combo_popularity
    
    # Ha én is játszom, akkor összesen hányan játsszuk
    # (statisztikailag én is "része" vagyok a combo_popularity-nak, de gyakorlatilag +1)
    total_players_with_combo = expected_players_with_combo + 1
    
    # FONTOS JAVÍTÁS: Az expected_other_winners azt jelenti, hogy
    # "rajtam kívül hányan nyernének", de valójában a jackpot oszlik közöttünk
    # Ha 5-öt találok, akkor BIZTOSAN nyertem, a kérdés: hányan osztják meg velem?
    expected_co_winners = expected_players_with_combo  # rajtam kívül
    
    # Jackpot felosztva
    jackpot_total = estimated_total_jackpot * jackpot_multiplier
    jackpot_per_winner = jackpot_total / (expected_co_winners + 1)
    
    # Várható érték komponensek (szelvényár többszöröse)
    ev_5 = P_5 * jackpot_per_winner
    ev_4 = P_4 * avg_prize_4
    ev_3 = P_3 * avg_prize_3
    ev_2 = P_2 * avg_prize_2
    
    total_ev = ev_5 + ev_4 + ev_3 + ev_2 - 1.0  # -1 = szelvényár
    
    result = {
        'ev': total_ev,
        'ev_5': ev_5,
        'ev_4': ev_4,
        'ev_3': ev_3,
        'ev_2': ev_2,
        'combo_popularity': combo_popularity,
        'expected_co_winners': expected_co_winners,
        'expected_players_with_combo': expected_players_with_combo,
        'jackpot_per_winner': jackpot_per_winner,
        'jackpot_total': jackpot_total,
    }
    
    if verbose:
        print(f"\nKombináció: {numbers}")
        print(f"  Népszerűség: {combo_popularity:.4f}×")
        print(f"  Várható játékosok ezzel a kombóval: {expected_players_with_combo:.6f}")
        print(f"  Ha nyerek, társnyertesek: {expected_co_winners:.6f}")
        print(f"  Jackpot összesen: {jackpot_total:.1f}×")
        print(f"  Jackpot/fő: {jackpot_per_winner:.1f}×")
        print(f"  EV_5: {ev_5:.6f}×, EV_4: {ev_4:.6f}×, EV_3: {ev_3:.6f}×, EV_2: {ev_2:.6f}×")
        print(f"  → TELJES EV: {total_ev:.6f}× ({total_ev*400:.2f} Ft)")
    
    return result

# ============================================================================
# STRATÉGIÁK ÖSSZEHASONLÍTÁSA
# ============================================================================

print("\n" + "="*80)
print("VÁRHATÓ ÉRTÉK ÖSSZEHASONLÍTÁS")
print("="*80)

strategies = {
    'Legritkább számok': [87, 80, 88, 90, 75],  # Top 5 legkevésbé népszerű
    'Születésnapok': [7, 13, 19, 23, 31],       # Népszerű tartomány
    'Magas számok': [82, 85, 86, 87, 90],       # 81-90 tartomány
    'Vegyes optimalizált': [40, 62, 75, 87, 90], # Mix a legritkábbakból
    'Véletlen': [12, 28, 45, 63, 79],           # Kontroll
    'Legnépszerűbb': [5, 19, 73, 4, 9],         # Top 5 legnépszerűbb
}

avg_players = huzasok_clean['estimated_players'].mean()

print(f"\nÁtlagos játékosszám: {avg_players:,.0f} szelvény")
print(f"Becsült jackpot: {estimated_total_jackpot:.1f}× ({estimated_total_jackpot*400:,.0f} Ft)\n")

results = []
for name, numbers in strategies.items():
    result = calculate_ev(numbers, number_popularity, avg_players)
    results.append((name, numbers, result))
    
    print(f"{name}")
    print(f"  Számok: {numbers}")
    print(f"  Népszerűség: {result['combo_popularity']:.4f}×")
    print(f"  Várható társnyertesek (ha nyerek): {result['expected_co_winners']:.6f}")
    print(f"  Jackpot/fő: {result['jackpot_per_winner']:.1f}× ({result['jackpot_per_winner']*400:,.0f} Ft)")
    print(f"  EV komponensek:")
    print(f"    • 5 találat: {result['ev_5']:.6f}× ({result['ev_5']*400:.2f} Ft)")
    print(f"    • 4 találat: {result['ev_4']:.6f}× ({result['ev_4']*400:.2f} Ft)")
    print(f"    • 3 találat: {result['ev_3']:.6f}× ({result['ev_3']*400:.2f} Ft)")
    print(f"    • 2 találat: {result['ev_2']:.6f}× ({result['ev_2']*400:.2f} Ft)")
    print(f"  → TELJES EV: {result['ev']:.6f}× = {result['ev']*400:.2f} Ft/szelvény")
    print()

# Sorbarendezés EV szerint
results_sorted = sorted(results, key=lambda x: x[2]['ev'], reverse=True)

print("="*80)
print("RANGSOR (EV szerint):")
print("="*80)
for i, (name, numbers, result) in enumerate(results_sorted, 1):
    improvement_vs_worst = (result['ev'] - results_sorted[-1][2]['ev']) * 400
    print(f"{i}. {name:25s} | EV: {result['ev']*400:+7.2f} Ft | "
          f"Javulás: {improvement_vs_worst:+6.2f} Ft")

best = results_sorted[0]
worst = results_sorted[-1]
improvement = (best[2]['ev'] - worst[2]['ev']) * 400

print(f"\n→ LEGJOBB vs LEGROSSZABB különbség: {improvement:.2f} Ft/szelvény")
print(f"   ({improvement/400*100:.2f}% javulás a várható veszteségben)")



VÁRHATÓ ÉRTÉK KALKULÁCIÓ

Átlagos nyeremények (szelvényár többszöröse):
  4 találat: 7947.6× (kb 3,179,033 Ft mai áron)
  3 találat: 82.3× (kb 32,927 Ft mai áron)
  2 találat: 6.0× (kb 2,404 Ft mai áron)

--------------------------------------------------------------------------------
5-TALÁLAT JACKPOT BECSLÉSE
--------------------------------------------------------------------------------

Elemzés 2020 óta: 308 sorsolás
Átlagos játékosszám: 3,295,396

Átlagos kifizetés szelvényenként:
  2 találat: 0.149956× (14.9956%)
  3 találat: 0.048339× (4.8339%)
  4 találat: 0.045263× (4.5263%)
  2+3+4 összesen: 0.243559× (24.36%)

Ha a teljes kifizetési arány 50%:
  5-találatra marad: 0.256441× (25.64%)

→ Becsült átlagos teljes jackpot/sorsolás: 845074.1×
  (kb 338,029,628 Ft mai áron)

Validálás (25 sorsolás volt 5-találatossal):
  Megfigyelt átlag jackpot: 6896218.5×
  Átlagos nyertesszám: 1.12
  Különbség: 6051144.4× (716.0%)
  → Megfigyelt érték használata (elég nagy minta)

VÉGSŐ BECSLÉS